# Lab 14 — DensityMatrix, Ruido y Tomografía Guiada

Los estados cuánticos reales son **estados mixtos** descritos por matrices de densidad ρ.
Cuando hay ruido, decoherencia o medición parcial, el formalismo del vector de estado es insuficiente.

**Herramientas**: `DensityMatrix`, `partial_trace`, canales de ruido, tomografía de estado.

## 1. Matrices de densidad: puras vs mixtas

Un estado ρ es:
- **Puro**: ρ² = ρ, Tr(ρ²) = 1, rango 1
- **Mixto**: ρ² ≠ ρ, Tr(ρ²) < 1, rango > 1

La pureza P = Tr(ρ²) mide el grado de mezcla (P=1 puro, P=1/d maximalmente mixto).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix, Statevector, partial_trace, state_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# Estado puro |+⟩
dm_pure = DensityMatrix.from_label('+')
print('Estado puro |+⟩:')
print(f'  Tr(ρ²) = {dm_pure.purity():.4f} (debe ser 1)')
print(f'  Rango = {np.linalg.matrix_rank(dm_pure.data)}')

# Estado maximalmente mixto I/2
dm_mixed = DensityMatrix(np.eye(2) / 2)
print('\nEstado maximalmente mixto I/2:')
print(f'  Tr(ρ²) = {dm_mixed.purity():.4f} (debe ser 0.5)')
print(f'  Rango = {np.linalg.matrix_rank(dm_mixed.data)}')

# Mezcla interpolada: ρ(p) = p|+⟩⟨+| + (1-p)I/2
p_vals = np.linspace(0, 1, 50)
purities = []
for p in p_vals:
    rho = p * dm_pure.data + (1-p) * np.eye(2)/2
    dm = DensityMatrix(rho)
    purities.append(dm.purity())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_vals, purities, 'b-', linewidth=2)
ax.set_xlabel('p (fracción del estado puro)', fontsize=12)
ax.set_ylabel('Pureza Tr(ρ²)', fontsize=12)
ax.set_title('Pureza de la mezcla ρ(p) = p|+⟩⟨+| + (1-p)I/2', fontsize=13)
ax.axhline(0.5, color='r', linestyle='--', alpha=0.5, label='Mixto máximo')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Canal de ruido despolarizante

El canal despolarizante de parámetro p aplica X, Y, Z con probabilidad p/3 cada uno:

$$\mathcal{E}(\rho) = (1-p)\rho + \frac{p}{3}(X\rho X + Y\rho Y + Z\rho Z)$$

Para p=0: sin ruido. Para p=3/4: mezcla máxima I/2.

In [ ]:
def apply_depolarizing(rho_mat: np.ndarray, p: float) -> np.ndarray:
    """Canal despolarizante aplicado a matriz 2x2."""
    X = np.array([[0,1],[1,0]])
    Y = np.array([[0,-1j],[1j,0]])
    Z = np.array([[1,0],[0,-1]])
    return (1-p)*rho_mat + (p/3)*(X@rho_mat@X + Y@rho_mat@Y + Z@rho_mat@Z)

# Estado inicial |+⟩
rho0 = dm_pure.data

# Pureza tras n aplicaciones del canal
p_noise = 0.05  # 5% por aplicación
n_steps = 50
purities_noise = []
bloch_x = []  # componente X del vector de Bloch

rho = rho0.copy()
for _ in range(n_steps):
    purities_noise.append(DensityMatrix(rho).purity())
    bloch_x.append(2 * rho[0, 1].real)  # ⟨X⟩ = Tr(Xρ) = 2*Re(ρ₀₁)
    rho = apply_depolarizing(rho, p_noise)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(purities_noise, 'b-', linewidth=2)
axes[0].axhline(0.5, color='r', linestyle='--', label='Mixto máximo')
axes[0].set_xlabel('Número de aplicaciones'); axes[0].set_ylabel('Pureza')
axes[0].set_title('Decoherencia: pureza vs tiempo')
axes[0].legend()
axes[1].plot(bloch_x, 'g-', linewidth=2)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Número de aplicaciones'); axes[1].set_ylabel('⟨X⟩')
axes[1].set_title('Decoherencia: vector de Bloch')
plt.suptitle(f'Canal despolarizante p={p_noise}', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Simulación con ruido usando AerSimulator

Simulamos un circuito real con un modelo de ruido basado en errores de puerta. Comparamos el estado sin ruido vs con ruido.

In [ ]:
# Estado de Bell con y sin ruido
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# Estado ideal
sv_ideal = Statevector(qc)
dm_ideal = DensityMatrix(sv_ideal)

# Modelo de ruido: error despolarizante en CNOT
noise_model = NoiseModel()
error_cx = depolarizing_error(0.05, 2)  # 5% error en 2-qubit
error_h = depolarizing_error(0.01, 1)   # 1% error en 1-qubit
noise_model.add_all_qubit_quantum_error(error_cx, ['cx'])
noise_model.add_all_qubit_quantum_error(error_h, ['h'])

# Simulación con ruido (usando save_density_matrix)
qc_dm = qc.copy()
qc_dm.save_density_matrix()
sim = AerSimulator(noise_model=noise_model)
job = sim.run(qc_dm, shots=1)
result = job.result()
dm_noisy = DensityMatrix(result.data(0)['density_matrix'])

print(f'Fidelidad ideal vs ruidoso: {state_fidelity(dm_ideal, dm_noisy):.4f}')
print(f'Pureza ideal:   {dm_ideal.purity():.4f}')
print(f'Pureza ruidosa: {dm_noisy.purity():.4f}')

## 4. Tomografía de estado (State Tomography)

La tomografía de estado reconstruye ρ midiendo en las bases X, Y, Z para cada qubit.
Para n qubits se necesitan 3ⁿ configuraciones de medición.

In [ ]:
def tomography_1qubit(dm: DensityMatrix) -> np.ndarray:
    """Reconstruye la DM de 1 qubit desde medidas X,Y,Z."""
    # ⟨X⟩ = dm[0,1] + dm[1,0], ⟨Y⟩ = i(dm[0,1] - dm[1,0]), ⟨Z⟩ = dm[0,0] - dm[1,1]
    rho = dm.data
    exp_X = 2 * rho[0, 1].real
    exp_Y = -2 * rho[0, 1].imag
    exp_Z = rho[0, 0].real - rho[1, 1].real

    # Reconstrucción: ρ = (I + xX + yY + zZ) / 2
    X = np.array([[0,1],[1,0]])
    Y = np.array([[0,-1j],[1j,0]])
    Z = np.array([[1,0],[0,-1]])
    I = np.eye(2)
    rho_rec = (I + exp_X * X + exp_Y * Y + exp_Z * Z) / 2
    return rho_rec

# Verificar tomografía en varios estados
test_states = {
    '|0⟩': DensityMatrix.from_label('0'),
    '|+⟩': DensityMatrix.from_label('+'),
    '|+i⟩': DensityMatrix([[0.5, -0.5j], [0.5j, 0.5]]),
    'Mixto I/2': DensityMatrix(np.eye(2)/2),
}

print('Verificación de tomografía 1 qubit:')
print(f'{"Estado":<10} | Fidelidad reconstrucción')
print('-' * 40)
for name, dm in test_states.items():
    rho_rec = tomography_1qubit(dm)
    dm_rec = DensityMatrix(rho_rec)
    fid = state_fidelity(dm, dm_rec)
    print(f'{name:<10} | {fid:.6f}')

## 5. Resumen

| Concepto | Descripción | Herramienta Qiskit |
|----------|-------------|--------------------|
| Estado mixto | ρ con Tr(ρ²) < 1 | `DensityMatrix` |
| Pureza | Tr(ρ²) ∈ [1/d, 1] | `.purity()` |
| Canal de ruido | Mapa CPTP | `NoiseModel` |
| Traza parcial | ρ_A = Tr_B(ρ_AB) | `partial_trace()` |
| Tomografía | Reconstruir ρ desde medidas | Medidas X,Y,Z |

**Relación fundamental**: un estado puro bipartito |ψ⟩_AB tiene estado reducido mixto ρ_A si y solo si hay entrelazamiento. La pureza de ρ_A mide el entrelazamiento.

In [ ]:
# Demostración: entrelazamiento ↔ mezcla del estado reducido
from qiskit.quantum_info import entropy

print('Entrelazamiento y pureza del estado reducido:')
print(f'{"Estado":<15} | Pureza ρ_A | S(ρ_A)')
print('-' * 45)
for angle_deg in [0, 30, 45, 60, 90]:
    angle = np.radians(angle_deg)
    qc = QuantumCircuit(2)
    qc.ry(2*angle, 0)
    qc.cx(0, 1)
    dm_full = DensityMatrix(Statevector(qc))
    dm_A = partial_trace(dm_full, [1])
    pur = dm_A.purity()
    ent = entropy(dm_A, base=2)
    label = f'θ={angle_deg}°'
    print(f'{label:<15} | {pur:.4f}    | {ent:.4f}')